In [0]:
# ── Time Travel: reconstruct last month's payroll ─────────────────
from dateutil.relativedelta import relativedelta
from datetime import datetime

last_month = datetime.now() - relativedelta(months=1)
spark.sql(f"""
  SELECT * FROM hr_catalog_divansh.silver.employees_enriched
  TIMESTAMP AS OF '{last_month.strftime('%Y-%m-%d')}'
""").createOrReplaceTempView("last_month_payroll")

spark.sql("""
  SELECT department_name, SUM(total_monthly_compensation) AS payroll
  FROM last_month_payroll GROUP BY department_name
""").show()

# ── MERGE-based incremental salary load ───────────────────────────
spark.sql("""
  MERGE INTO hr_catalog_divansh.silver.employees_enriched AS target
  USING (
    SELECT e.employee_id, s.base_salary, s.bonus_pct, s.effective_date,
           s.base_salary + s.base_salary * s.bonus_pct AS total_monthly_compensation
    FROM hr_catalog_divansh.bronze.salaries s
    JOIN hr_catalog_divansh.bronze.employees e USING (employee_id)
    QUALIFY ROW_NUMBER() OVER (PARTITION BY s.employee_id ORDER BY s.effective_date DESC) = 1
  ) AS source
  ON target.employee_id = source.employee_id
  WHEN MATCHED AND (
    target.base_salary  <> source.base_salary OR
    target.bonus_pct    <> source.bonus_pct
  ) THEN UPDATE SET
    target.base_salary               = source.base_salary,
    target.bonus_pct                 = source.bonus_pct,
    target.total_monthly_compensation = source.total_monthly_compensation,
    target.effective_date            = source.effective_date
""")

# ── Python task: send payroll summary via Databricks Notifications API ──
import requests, json

summary = spark.sql("""
  SELECT department_name, total_monthly_payroll
  FROM hr_catalog_divansh.gold.dept_payroll_summary
""").toPandas().to_string(index=False)

token = dbutils.secrets.get("hr_scope", "db_token")
workspace = dbutils.notebook.entry_point.getDbutils().notebook().getContext().browserHostName().get()

requests.post(
    f"https://{workspace}/api/2.0/jobs/runs/submit",
    headers={"Authorization": f"Bearer {token}"},
    json={
        "run_name": "payroll_summary_notification",
        "tasks": [{
            "task_key": "notify",
            "notebook_task": {"notebook_path": "/Repos/.../send_report"},
            "libraries": []
        }]
    }
)
print("Payroll Summary:\n", summary)